# Carga de Datos

Carga datos transformados en archivos y bases de datos de forma segura

## Introducción

La carga es la fase final del pipeline ETL: escribir los datos transformados a su destino permanente. El destino puede ser archivos (CSV, JSON, Excel), bases de datos (SQLite, PostgreSQL) o sistemas en la nube. Una carga bien diseñada es idempotente — ejecutarla dos veces produce el mismo resultado que ejecutarla una vez. Esto requiere validar los datos antes de escribirlos, elegir correctamente entre replace/append, usar inserciones en lote para rendimiento, y verificar la integridad después de la carga.

### Objetivos de Aprendizaje

- Escribir DataFrames a CSV, JSON y Excel con pandas
- Cargar datos en SQLite usando to_sql() con if_exists="replace"/"append"
- Implementar inserciones en lote para mejor rendimiento con grandes volúmenes
- Validar datos antes de la carga: tipos, rangos, valores requeridos
- Diseñar cargas idempotentes que no dupliquen datos al re-ejecutarse

## Escribir a CSV, JSON y Excel

> pandas ofrece to_csv(), to_json() y to_excel() para escribir a los formatos de archivo más comunes. Parámetros importantes: index=False para no incluir el índice numérico del DataFrame, encoding para archivos con caracteres especiales, y orient en JSON para controlar la estructura del output. Para Excel, ExcelWriter permite escribir múltiples hojas en un mismo archivo.

In [ ]:
import pandas as pd
from pathlib import Path

df = pd.DataFrame({
    'id':         [1, 2, 3, 4, 5],
    'producto':   ['Laptop', 'Mouse', 'Monitor', 'Teclado', 'Cámara'],
    'precio':     [1200.0, 25.5, 350.0, 45.0, 89.99],
    'categoria':  ['Electrónica', 'Periféricos', 'Electrónica', 'Periféricos', 'Electrónica'],
    'en_stock':   [True, True, False, True, True]
})

df.to_csv('productos.csv', index=False)
print(f"✓ CSV escrito: {len(df)} filas")

df.to_json('productos.json', orient='records', indent=2, force_ascii=False)
print(f"✓ JSON escrito: {len(df)} registros")

df.to_excel('productos.xlsx', sheet_name='Inventario', index=False)
print(f"✓ Excel escrito: {len(df)} filas")

df_electronica = df[df['categoria'] == 'Electrónica']
df_perifericos  = df[df['categoria'] == 'Periféricos']

with pd.ExcelWriter('reporte.xlsx', engine='openpyxl') as writer:
    df.to_excel(writer, sheet_name='Todos', index=False)
    df_electronica.to_excel(writer, sheet_name='Electrónica', index=False)
    df_perifericos.to_excel(writer, sheet_name='Periféricos', index=False)

print(f"✓ Reporte multi-hoja escrito")

for f in ['productos.csv', 'productos.json', 'productos.xlsx', 'reporte.xlsx']:
    Path(f).unlink(missing_ok=True)

## Carga en SQLite con to_sql()

> pd.DataFrame.to_sql() escribe directamente a una tabla SQL. El parámetro if_exists controla el comportamiento cuando la tabla ya existe: "replace" la elimina y recrea (ideal para cargas completas diarias), "append" agrega filas (ideal para cargas incrementales), "fail" lanza error si existe. Siempre usa el context manager (with sqlite3.connect()) para garantizar que la conexión se cierre correctamente incluso si hay errores.

In [ ]:
import pandas as pd
import sqlite3

ventas = pd.DataFrame({
    'id':          [1, 2, 3, 4, 5],
    'fecha':       ['2024-01-15', '2024-01-15', '2024-01-16', '2024-01-17', '2024-01-18'],
    'producto':    ['Laptop', 'Mouse', 'Monitor', 'Teclado', 'Laptop'],
    'cantidad':    [2, 10, 3, 5, 1],
    'precio':      [1200.0, 25.5, 350.0, 45.0, 1200.0],
    'total':       [2400.0, 255.0, 1050.0, 225.0, 1200.0]
})

with sqlite3.connect(':memory:') as conn:
    ventas.to_sql('ventas', conn, if_exists='replace', index=False)
    n = pd.read_sql("SELECT COUNT(*) AS n FROM ventas", conn).iloc[0, 0]
    print(f"✓ replace: {n} filas en tabla 'ventas'")

ventas_nuevas = pd.DataFrame({
    'id':       [6, 7],
    'fecha':    ['2024-01-19', '2024-01-19'],
    'producto': ['Cámara', 'Mouse'],
    'cantidad': [1, 3],
    'precio':   [89.99, 25.5],
    'total':    [89.99, 76.5]
})

with sqlite3.connect(':memory:') as conn:
    conn.execute("CREATE TABLE ventas (id INT, fecha TEXT, producto TEXT, cantidad INT, precio REAL, total REAL)")
    ventas.to_sql('ventas', conn, if_exists='append', index=False)
    ventas_nuevas.to_sql('ventas', conn, if_exists='append', index=False)
    n = pd.read_sql("SELECT COUNT(*) AS n FROM ventas", conn).iloc[0, 0]
    print(f"✓ append: ahora {n} filas")

    df_resultado = pd.read_sql(
        "SELECT producto, SUM(total) AS ingresos FROM ventas GROUP BY producto",
        conn
    )
    print("\nIngresos por producto:")
    print(df_resultado.to_string(index=False))

## Inserciones en Lote para Rendimiento

> Para grandes volúmenes de datos, una inserción por fila es muy lenta (N roundtrips a la BD). Las inserciones en lote (batch inserts) agrupan muchas filas en una sola operación SQL, reduciendo dramáticamente el tiempo. to_sql() acepta el parámetro chunksize para controlar el tamaño de los lotes. Para máximo rendimiento con SQLite, también envuelve todo en una transacción explícita (BEGIN / COMMIT).

In [ ]:
import pandas as pd
import sqlite3
import time
import numpy as np

N = 100_000
np.random.seed(42)
df_grande = pd.DataFrame({
    'id':        range(1, N + 1),
    'fecha':     pd.date_range('2024-01-01', periods=N, freq='h').strftime('%Y-%m-%d %H:%M'),
    'valor':     np.random.uniform(10, 1000, N).round(2),
    'categoria': np.random.choice(['A', 'B', 'C', 'D'], N),
    'activo':    np.random.choice([True, False], N)
})
print(f"Dataset de {len(df_grande):,} filas generado")

inicio = time.time()
with sqlite3.connect(':memory:') as conn:
    df_grande.to_sql('datos', conn, if_exists='replace', index=False)
    n = pd.read_sql("SELECT COUNT(*) AS n FROM datos", conn).iloc[0,0]
t1 = time.time() - inicio
print(f"Sin chunksize  : {t1:.3f}s | {n:,} filas")

inicio = time.time()
with sqlite3.connect(':memory:') as conn:
    df_grande.to_sql('datos', conn, if_exists='replace', index=False, chunksize=10_000)
    n = pd.read_sql("SELECT COUNT(*) AS n FROM datos", conn).iloc[0,0]
t2 = time.time() - inicio
print(f"chunksize=10k  : {t2:.3f}s | {n:,} filas")

inicio = time.time()
conn = sqlite3.connect(':memory:')
conn.execute("CREATE TABLE datos (id INT, fecha TEXT, valor REAL, categoria TEXT, activo INT)")
conn.execute("BEGIN")
filas = list(df_grande.itertuples(index=False, name=None))
conn.executemany("INSERT INTO datos VALUES (?,?,?,?,?)", filas)
conn.commit()
n = conn.execute("SELECT COUNT(*) FROM datos").fetchone()[0]
conn.close()
t3 = time.time() - inicio
print(f"executemany    : {t3:.3f}s | {n:,} filas")

## Validación de Datos Antes de Cargar

> Validar antes de cargar garantiza que nunca lleguen datos corruptos o incompletos al destino. Las validaciones típicas son: columnas requeridas presentes, tipos de datos correctos, rangos válidos (precio > 0, edad < 150), valores únicos donde se requiere (IDs sin duplicados), y porcentaje de nulos por debajo de un umbral. Un pipeline que valida y rechaza datos incorrectos es mucho mejor que uno que carga silenciosamente datos de baja calidad.

In [ ]:
import pandas as pd
import numpy as np
from dataclasses import dataclass, field
from typing import Any

@dataclass
class ResultadoValidacion:
    valido: bool = True
    errores: list = field(default_factory=list)
    advertencias: list = field(default_factory=list)

def validar_dataframe(df: pd.DataFrame,
                       columnas_requeridas: list[str],
                       tipos_esperados: dict[str, type],
                       rangos: dict[str, tuple[Any, Any]],
                       max_pct_nulos: float = 0.10) -> ResultadoValidacion:
    resultado = ResultadoValidacion()

    faltantes = set(columnas_requeridas) - set(df.columns)
    if faltantes:
        resultado.errores.append(f"Columnas faltantes: {faltantes}")
        resultado.valido = False

    if len(df) == 0:
        resultado.errores.append("El DataFrame está vacío")
        resultado.valido = False
        return resultado

    for col, tipo in tipos_esperados.items():
        if col in df.columns:
            if not pd.api.types.is_dtype_equal(df[col].dtype, tipo):
                resultado.advertencias.append(
                    f"Columna '{col}': tipo esperado {tipo}, encontrado {df[col].dtype}")

    for col in columnas_requeridas:
        if col in df.columns:
            pct_nulos = df[col].isna().mean()
            if pct_nulos > max_pct_nulos:
                resultado.errores.append(
                    f"Columna '{col}': {pct_nulos:.1%} nulos (máx permitido: {max_pct_nulos:.0%})")
                resultado.valido = False

    for col, (minimo, maximo) in rangos.items():
        if col in df.columns:
            col_sin_nulos = df[col].dropna()
            fuera_rango = ((col_sin_nulos < minimo) | (col_sin_nulos > maximo)).sum()
            if fuera_rango > 0:
                resultado.errores.append(
                    f"Columna '{col}': {fuera_rango} valores fuera del rango [{minimo}, {maximo}]")
                resultado.valido = False

    if 'id' in df.columns:
        n_dup = df['id'].duplicated().sum()
        if n_dup > 0:
            resultado.errores.append(f"{n_dup} IDs duplicados encontrados")
            resultado.valido = False

    return resultado

df_test = pd.DataFrame({
    'id':       [1, 2, 2, 4, 5],
    'producto': ['Laptop', 'Mouse', 'Mouse', 'Monitor', None],
    'precio':   [1200.0, 25.5, 25.5, 350.0, -99.0],
    'cantidad': [2, 10, 10, 3, 5]
})

val = validar_dataframe(
    df_test,
    columnas_requeridas=['id', 'producto', 'precio'],
    tipos_esperados={'precio': 'float64', 'cantidad': 'int64'},
    rangos={'precio': (0, 10000), 'cantidad': (1, 999)},
    max_pct_nulos=0.10
)

print(f"¿DataFrame válido?: {val.valido}")
if val.errores:
    print("Errores de validación:")
    for e in val.errores:
        print(f"   • {e}")
if val.advertencias:
    print("Advertencias:")
    for a in val.advertencias:
        print(f"   • {a}")

## Cargas Idempotentes: Sin Duplicados al Re-ejecutar

> Una carga idempotente produce el mismo resultado independientemente de cuántas veces se ejecute. Es crucial para pipelines que se re-ejecutan ante fallos. El patrón básico: antes de insertar, verifica qué ya existe en el destino usando una clave única (id, fecha+producto, etc.), y solo inserta los registros nuevos. Alternativas: UPSERT en SQL (INSERT OR REPLACE en SQLite) o if_exists="replace" para tablas que se reescriben completamente.

In [ ]:
import pandas as pd
import sqlite3

def cargar_incremental(df_nuevos: pd.DataFrame, db_path: str,
                        tabla: str, col_clave: str) -> dict:
    with sqlite3.connect(db_path) as conn:
        try:
            ids_existentes = set(
                pd.read_sql(f"SELECT {col_clave} FROM {tabla}", conn)[col_clave]
            )
        except Exception:
            ids_existentes = set()

        df_realmente_nuevos = df_nuevos[~df_nuevos[col_clave].isin(ids_existentes)]

        if len(df_realmente_nuevos) == 0:
            print(f"✓ Sin datos nuevos — {len(df_nuevos)} ya existían")
            return {'insertados': 0, 'ya_existian': len(df_nuevos)}

        df_realmente_nuevos.to_sql(tabla, conn, if_exists='append', index=False)
        print(f"✓ Insertados {len(df_realmente_nuevos)} registros nuevos "
              f"(se ignoraron {len(df_nuevos) - len(df_realmente_nuevos)} duplicados)")
        return {'insertados': len(df_realmente_nuevos),
                'ya_existian': len(df_nuevos) - len(df_realmente_nuevos)}

def upsert_sqlite(df: pd.DataFrame, db_path: str, tabla: str) -> None:
    conn = sqlite3.connect(db_path)
    try:
        placeholders = ', '.join(['?'] * len(df.columns))
        sql = f"INSERT OR REPLACE INTO {tabla} VALUES ({placeholders})"
        conn.execute("BEGIN")
        conn.executemany(sql, df.itertuples(index=False, name=None))
        conn.commit()
        print(f"✓ UPSERT completado: {len(df)} filas")
    except Exception as e:
        conn.rollback()
        raise
    finally:
        conn.close()

df_ventas = pd.DataFrame({
    'id': [1, 2, 3, 4, 5],
    'producto': ['Laptop','Mouse','Monitor','Teclado','Cámara'],
    'total': [2400.0, 255.0, 1050.0, 225.0, 90.0]
})

resultado1 = cargar_incremental(df_ventas, ':memory:', 'ventas', 'id')

df_mas_ventas = pd.DataFrame({
    'id': [3, 4, 6, 7],
    'producto': ['Monitor','Teclado','Hub USB','Webcam'],
    'total': [1050.0, 225.0, 35.0, 65.0]
})

resultado2 = cargar_incremental(df_mas_ventas, ':memory:', 'ventas', 'id')

## Pipeline de Carga Completo con Validación

Pipeline de carga que valida los datos, usa inserciones en lote, verifica la carga, y genera un reporte de ejecución.

In [ ]:
import pandas as pd
import sqlite3
import logging
import sys

logging.basicConfig(level=logging.INFO, format='%(levelname)-8s | %(message)s',
                    stream=sys.stdout)
logger = logging.getLogger("carga_pipeline")

ventas = pd.DataFrame({
    'id':       range(1, 11),
    'fecha':    ['2024-01-15']*5 + ['2024-01-16']*5,
    'producto': ['Laptop','Mouse','Monitor','Teclado','Cámara',
                  'Laptop','Mouse','Monitor','Teclado','Hub'],
    'cantidad': [2, 10, 3, 5, 1, 3, 15, 2, 8, 4],
    'precio':   [1200.0, 25.5, 350.0, 45.0, 89.99,
                  1200.0, 25.5, 350.0, 45.0, 29.99],
    'total':    [2400.0, 255.0, 1050.0, 225.0, 89.99,
                  3600.0, 382.5, 700.0, 360.0, 119.96],
    'region':   ['Norte','Sur','Norte','Este','Sur',
                  'Norte','Oeste','Sur','Norte','Este']
})

def validar_antes_de_cargar(df: pd.DataFrame) -> tuple[bool, list[str]]:
    errores = []

    if len(df) == 0:
        errores.append("DataFrame vacío")
        return False, errores

    cols_req = {'id', 'fecha', 'producto', 'cantidad', 'precio', 'total'}
    faltantes = cols_req - set(df.columns)
    if faltantes:
        errores.append(f"Columnas faltantes: {faltantes}")

    nulos_criticos = df[['id', 'producto', 'precio']].isna().sum()
    for col, n in nulos_criticos.items():
        if n > 0:
            errores.append(f"'{col}' tiene {n} valores nulos")

    if 'precio' in df.columns:
        invalidos = (df['precio'] <= 0).sum()
        if invalidos > 0:
            errores.append(f"precio: {invalidos} valores <= 0")

    if 'id' in df.columns:
        dup = df['id'].duplicated().sum()
        if dup > 0:
            errores.append(f"{dup} IDs duplicados")

    return len(errores) == 0, errores

def cargar_con_validacion(df: pd.DataFrame, db_path: str = ':memory:') -> dict:
    logger.info("INICIANDO CARGA DE DATOS")
    reporte = {'exito': False, 'filas_cargadas': 0, 'errores': [], 'advertencias': []}

    logger.info(f"Validando {len(df)} filas...")
    valido, errores = validar_antes_de_cargar(df)
    if not valido:
        reporte['errores'] = errores
        for e in errores:
            logger.error(f"  ✗ {e}")
        logger.error("Carga CANCELADA por errores de validación")
        return reporte

    logger.info("  ✓ Validación pasada")

    conn = sqlite3.connect(db_path)
    try:
        conn.execute("BEGIN")
        df.to_sql('ventas', conn, if_exists='replace', index=False, chunksize=1000)
        conn.commit()
        logger.info(f"  ✓ Datos escritos en lotes de 1000")
    except Exception as e:
        conn.rollback()
        reporte['errores'].append(f"Error de escritura: {e}")
        logger.error(f"Error en carga, rollback ejecutado: {e}")
        conn.close()
        return reporte

    n_cargadas = pd.read_sql("SELECT COUNT(*) AS n FROM ventas", conn).iloc[0,0]
    if n_cargadas != len(df):
        reporte['advertencias'].append(
            f"Discrepancia: esperadas {len(df)}, encontradas {n_cargadas}")
        logger.warning(f"  ⚠ Discrepancia de filas: {n_cargadas} vs {len(df)}")
    else:
        logger.info(f"  ✓ Verificación OK: {n_cargadas} filas confirmadas")

    resumen = pd.read_sql("""
        SELECT region, COUNT(*) AS transacciones, SUM(total) AS ingresos
        FROM ventas GROUP BY region ORDER BY ingresos DESC
    """, conn)
    conn.close()

    reporte['exito'] = True
    reporte['filas_cargadas'] = n_cargadas
    logger.info("CARGA COMPLETADA EXITOSAMENTE")

    print("\n=== RESUMEN POST-CARGA ===")
    print(resumen.to_string(index=False))
    return reporte

resultado = cargar_con_validacion(ventas)
print(f"\n=== REPORTE DE CARGA ===")
print(f"  Éxito      : {resultado['exito']}")
print(f"  Filas      : {resultado['filas_cargadas']}")

## Tips y Mejores Prácticas

> Siempre verifica el resultado de to_sql() con una consulta SELECT COUNT(*) después de la carga. to_sql() no lanza error si el DataFrame estaba vacío — simplemente no carga nada. Sin verificación, tu pipeline puede reportar éxito con una tabla vacía.

> if_exists="replace" elimina y recrea la tabla INCLUYENDO sus índices, constraints y permisos definidos en SQL. Si la tabla tiene índices personalizados para rendimiento, deberás recrearlos después de cada carga. Considera "append" + DELETE previo para tablas con estructura compleja.

> Para archivos CSV con caracteres especiales (acentos, ñ) que se abrirán en Excel, usa encoding="utf-8-sig" en to_csv(). El BOM (Byte Order Mark) le indica a Excel que el archivo es UTF-8 y lo abre correctamente sin mostrar caracteres extraños.

> Diseña tus cargas como idempotentes desde el principio: ejecutar el mismo pipeline dos veces debe producir el mismo estado final. Esto simplifica enormemente el manejo de fallos — si algo falla, simplemente re-ejecutas sin miedo a duplicar datos.

## Errores Comunes

### Usar if_exists="append" sin verificar duplicados

¿Por qué ocurre?
- Si el pipeline falla a mitad y se re-ejecuta, append insertará nuevamente los mismos registros, creando duplicados. Esto corrompe el dataset y puede ser difícil de detectar.

Solución
- Antes de usar append, verifica qué IDs ya existen en la tabla destino y filtra el DataFrame para insertar solo los nuevos. O usa INSERT OR REPLACE (UPSERT) de SQLite para actualizar si el ID ya existe.

### Cargar datos sin validación previa

¿Por qué ocurre?
- Cargar datos con precios negativos, IDs duplicados, o columnas mal tipadas produce datos corruptos en producción. El problema se descubre tarde (cuando alguien nota un reporte incorrecto) y es difícil de trazar hasta la carga.

Solución
- Implementa una función de validación que se ejecute ANTES de to_sql(). Verifica: tipos de datos, rangos válidos, nulos en columnas clave, y ausencia de duplicados en columnas ID.

### No cerrar la conexión SQLite después de usar to_sql()

¿Por qué ocurre?
- SQLite mantiene un lock en el archivo de base de datos mientras la conexión esté abierta. Si no cierras la conexión, otros procesos no pueden escribir en la BD, causando errores de "database is locked".

Solución
- Usa siempre el context manager: with sqlite3.connect(ruta) as conn: — garantiza el cierre automático incluso si hay excepciones. Alternativamente, llama a conn.close() en el bloque finally.

### Escribir archivos grandes sin verificar espacio en disco

¿Por qué ocurre?
- df.to_csv() para un DataFrame de 10M filas puede generar un archivo de varios GB. Si el disco se llena a mitad, el archivo queda truncado y corrompido.

Solución
- Para DataFrames grandes, usa escritura en chunks o comprime directamente: df.to_csv("archivo.csv.gz", compression="gzip").